### OCR Benchmark Evaluation — OmniDocBench JASyn

Đánh giá 3 model OCR (**`easyocr`**, **`PP-OCRv6_medium`**, **`Surya OCR`**) trên **518 trang tiếng Nhật (OmniDocBench JASyn)** (cố định Layout = `DocLayout-YOLO`, `OcrMode.FULL_PAGE`, Dynamic Image Scaling; chuẩn hóa `NFKC` + bỏ khoảng trắng thừa).

* **`NED` *(Normalized Edit Distance)*:** Khoảng cách chỉnh sửa chuẩn hóa (`Levenshtein(pred, gt) / max(|pred|, |gt|)`), càng thấp càng tốt.
* **`Similarity` *(`1 - NED`)*:** Độ tương đồng ký tự theo phần trăm (`(1 - NED) × 100%`), càng cao càng tốt.
* **`Micro CER` *(Character Error Rate)*:** Tỷ lệ lỗi ký tự tổng trên toàn tập (`∑ Levenshtein / ∑ |gt| × 100%`), càng thấp càng tốt.
* **`Len Ratio (Pred/GT)`:** Tỷ lệ tổng số ký tự dự đoán trên đáp án (`≈ 1.0` là cân bằng; `< 1` đọc sót, `> 1` đọc thừa).

In [1]:
import json
import re
import unicodedata
from pathlib import Path

import pandas as pd
from IPython.display import display

BASE_DIR = Path(".").resolve()
GT_PATH = BASE_DIR / "ground_truth.json"
MODELS = ["easyocr", "PP-OCRv6_medium", "Surya OCR"]
OUT_OF_SCOPE_LAYOUTS = {"book_vertical", "newspaper_vertical", "handwritten_note"}


def levenshtein_distance(a: str, b: str) -> int:
    """Exact Levenshtein distance via Myers' bit-parallel algorithm."""
    if a == b:
        return 0
    if not a:
        return len(b)
    if not b:
        return len(a)
    if len(a) > len(b):
        a, b = b, a
    m = len(a)
    peq: dict[str, int] = {}
    for i, ch in enumerate(a):
        peq[ch] = peq.get(ch, 0) | (1 << i)
    pv = (1 << m) - 1
    mv = 0
    score = m
    hb = 1 << (m - 1)
    mask = (1 << m) - 1
    for ch in b:
        eq = peq.get(ch, 0)
        xv = eq | mv
        xh = (((eq & pv) + pv) ^ pv) | eq
        ph = mv | (~(xh | pv) & mask)
        mh = pv & xh
        if ph & hb:
            score += 1
        elif mh & hb:
            score -= 1
        ph = ((ph << 1) | 1) & mask
        mh = (mh << 1) & mask
        pv = mh | (~(xv | ph) & mask)
        mv = ph & xv
    return score


def normalize_ja(text: str) -> str:
    text = unicodedata.normalize("NFKC", text or "")
    return re.sub(r"\s+", "", text)


def extract_gt_text(page: dict) -> str:
    items = []
    for d in page.get("layout_dets", []):
        if d.get("ignore"):
            continue
        txt = (d.get("text") or "").strip()
        if not txt:
            continue
        order = d.get("order")
        poly = d.get("poly") or [0] * 8
        key = (0, order) if order is not None else (1, min(poly[1::2]), min(poly[0::2]))
        items.append((key, txt))
    items.sort(key=lambda x: x[0])
    return normalize_ja("\n".join(t for _, t in items))


def extract_pred_text(doc: dict) -> str:
    parts = []
    for t in doc.get("texts", []):
        txt = (t.get("text") or "").strip()
        if txt:
            parts.append(txt)
    return normalize_ja("\n".join(parts))


gt_list = json.loads(GT_PATH.read_text(encoding="utf-8"))
gt_map = {Path(p["page_info"]["image_path"]).stem: p for p in gt_list}

records = []
for model in MODELS:
    m_dir = BASE_DIR / model
    for f in sorted(m_dir.glob("*.json")):
        if f.name in {"errors.json", "current_files.json"}:
            continue
        page = gt_map.get(f.stem)
        if page is None:
            continue
        gt_str = extract_gt_text(page)
        if not gt_str:
            continue
        pred_doc = json.loads(f.read_text(encoding="utf-8"))
        pred_str = extract_pred_text(pred_doc)
        ed = levenshtein_distance(pred_str, gt_str)
        ned = ed / max(len(pred_str), len(gt_str))
        layout_type = page.get("page_info", {}).get("page_attribute", {}).get("layout", "unknown")
        scope = "Out-of-Scope (Vertical/Handwritten)" if layout_type in OUT_OF_SCOPE_LAYOUTS else "In-Scope (Horizontal)"
        records.append({
            "model": model,
            "file": f.stem,
            "layout": layout_type,
            "scope": scope,
            "gt_len": len(gt_str),
            "pred_len": len(pred_str),
            "ed": ed,
            "ned": ned,
            "sim": (1.0 - ned) * 100.0,
        })

df = pd.DataFrame(records)
print(f"Evaluated {df['file'].nunique()} pages x {df['model'].nunique()} models = {len(df)} rows")

Evaluated 518 pages x 3 models = 1554 rows


In [2]:
# Table 1: Overall OCR Performance (All 518 Japanese pages)
overall_rows = []
for model in MODELS:
    sub = df[df["model"] == model]
    overall_rows.append({
        "Model": model,
        "Pages": len(sub),
        "NED (↓)": round(sub["ned"].mean(), 4),
        "Similarity 1-NED % (↑)": round(sub["sim"].mean(), 2),
        "Micro CER % (↓)": round(sub["ed"].sum() / sub["gt_len"].sum() * 100.0, 2),
        "Len Ratio (Pred/GT)": round(sub["pred_len"].sum() / sub["gt_len"].sum(), 3),
    })
overall_df = pd.DataFrame(overall_rows).sort_values("NED (↓)").reset_index(drop=True)
display(overall_df)

,Model,Pages,NED (↓),Similarity 1-NED % (↑),Micro CER % (↓),Len Ratio (Pred/GT)
0,PP-OCRv6_medium,518,0.4482,55.18,50.16,1.196
1,Surya OCR,518,0.4921,50.79,67.63,1.428
2,easyocr,518,0.8555,14.45,85.06,0.225


In [3]:
# Table 2: In-Scope (Horizontal) vs Out-of-Scope (Vertical / Handwritten)
scope_rows = []
for scope in ["In-Scope (Horizontal)", "Out-of-Scope (Vertical/Handwritten)"]:
    for model in MODELS:
        sub = df[(df["scope"] == scope) & (df["model"] == model)]
        scope_rows.append({
            "Scope": scope,
            "Model": model,
            "Pages": len(sub),
            "NED (↓)": round(sub["ned"].mean(), 4),
            "Similarity 1-NED % (↑)": round(sub["sim"].mean(), 2),
            "Micro CER % (↓)": round(sub["ed"].sum() / sub["gt_len"].sum() * 100.0, 2),
        })
scope_df = pd.DataFrame(scope_rows)
display(scope_df)

,Scope,Model,Pages,NED (↓),Similarity 1-NED % (↑),Micro CER % (↓)
0,In-Scope (Horizontal),easyocr,407,0.8312,16.88,83.49
1,In-Scope (Horizontal),PP-OCRv6_medium,407,0.3765,62.35,44.93
2,In-Scope (Horizontal),Surya OCR,407,0.4281,57.19,60.35
3,Out-of-Scope (Vertical/Handwritten),easyocr,111,0.9444,5.56,94.07
4,Out-of-Scope (Vertical/Handwritten),PP-OCRv6_medium,111,0.7114,28.86,80.07
5,Out-of-Scope (Vertical/Handwritten),Surya OCR,111,0.7267,27.33,109.27


In [4]:
# Table 3: Per Japanese PDF Type (14 layout types)
counts = df[df["model"] == MODELS[0]].groupby("layout")["file"].count()
piv_sim = df.pivot_table(index="layout", columns="model", values="sim", aggfunc="mean")[MODELS].round(2)
piv_ned = df.pivot_table(index="layout", columns="model", values="ned", aggfunc="mean")[MODELS].round(4)

per_type = pd.DataFrame({"Pages": counts})
for m in MODELS:
    per_type[f"{m} Sim% (↑)"] = piv_sim[m]
for m in MODELS:
    per_type[f"{m} NED (↓)"] = piv_ned[m]
per_type = per_type.sort_values("Pages", ascending=False)
display(per_type)

,Pages,easyocr Sim% (↑),PP-OCRv6_medium Sim% (↑),Surya OCR Sim% (↑),easyocr NED (↓),PP-OCRv6_medium NED (↓),Surya OCR NED (↓)
layout,,,,,,,
book_vertical,55,1.15,14.40,14.47,0.9885,0.8560,0.8553
text_book,53,11.87,61.94,56.34,0.8813,0.3806,0.4366
report,42,16.05,58.45,55.53,0.8395,0.4155,0.4447
slide,38,17.71,46.74,40.32,0.8229,0.5326,0.5968
press_release,38,19.22,64.61,59.21,0.8078,0.3539,0.4079
magazine,37,17.59,65.07,54.01,0.8241,0.3493,0.4599
user_manual,37,21.28,59.40,53.92,0.7872,0.4060,0.4608
two_column_simple,36,29.16,76.55,75.06,0.7084,0.2345,0.2494
academic_paper,36,6.42,66.01,64.26,0.9358,0.3399,0.3574
